In [ ]:
import pandas as pd
from io import StringIO
from urllib.parse import urlparse

import re
import emoji
import gc

In [ ]:
bad_lines = []

def bad_line_handler(line):
    bad_lines.append(line)
    return None  # Zeile überspringen

chunks = pd.read_csv(
    "bitcoin-tweets-2021.csv",
    chunksize=100_000,
    engine="python",          # 🔴 extrem wichtig
    on_bad_lines=bad_line_handler
)

In [ ]:
print(f"Anzahl bad lines: {len(bad_lines)}")

Anzahl bad lines: 0


In [ ]:
df_raw = pd.concat(chunks)

In [ ]:
del chunks
del bad_lines
gc.collect()

30

In [ ]:
df_raw.columns

Index(['datetime', 'username', 'text'], dtype='object')

In [ ]:
len(df_raw)

15630696

In [ ]:
wrong_text_rows = df_raw[~df_raw["text"].apply(lambda x: isinstance(x, str))]

In [ ]:
wrong_text_rows

,datetime,username,text
891075,https://t.co/u3iraDb6VV,None,None
891076,#ITintheD383,None,None
1134125,https://t.co/fRl6ZV3d8Z,None,None
1134126,#ITintheD384,None,None
1416485,#FelizTarde 👉 https://t.co/i9zJ57T14Y,None,None
...,...,...,...
14939818,#trading #emprende #FelizSabado 👉 https://t....,None,None
15483967,hot bitcoin roomba,None,None
15483968,tremble traffic expressly,None,None
15483969,the buggy brickwork grows this concrete,None,None


In [ ]:
wrong_text_ids = wrong_text_rows.index.tolist()
print("Zeilen mit falschem Typ:", wrong_text_ids)

Zeilen mit falschem Typ: [891075, 891076, 1134125, 1134126, 1416485, 3296300, 3296301, 3296302, 3296303, 3296304, 3296305, 3296306, 3296307, 3296308, 3296309, 3296310, 3296311, 3407877, 3407878, 3407879, 3407880, 3407881, 3407882, 3407883, 3407884, 3407885, 3407886, 3407887, 3407888, 3410683, 3410684, 3410685, 3410686, 3410687, 3410688, 3410689, 3410690, 3410691, 3410692, 3410693, 3410694, 3646111, 3646112, 3884318, 4102517, 4831633, 4831634, 5067033, 5067034, 5067035, 5067036, 5067037, 5181241, 5462725, 5800894, 5831097, 6039323, 6039324, 6673666, 6674412, 6676961, 6716830, 6752594, 6755338, 6828536, 6847565, 7057852, 7324540, 7398073, 7424834, 7556694, 7556695, 7685607, 7686349, 7768062, 8111045, 8111046, 8111047, 8126279, 8126280, 8126281, 8612561, 8830967, 8907561, 8925904, 8965799, 9283449, 9283450, 9283451, 9283452, 9506944, 9562014, 9684018, 9726205, 9857644, 9902884, 9916874, 9916875, 9917167, 9917168, 9973498, 9976697, 10132626, 10150111, 10150112, 10199673, 10219080, 10219081

In [ ]:
df_raw["text"].apply(type).value_counts()

,count
text,
<class 'str'>,15630309
<class 'NoneType'>,387


In [ ]:
wrong_username_rows = df_raw[~df_raw["username"].apply(lambda x: isinstance(x, str))]

In [ ]:
wrong_username_rows

,datetime,username,text
891075,https://t.co/u3iraDb6VV,None,None
891076,#ITintheD383,None,None
1134125,https://t.co/fRl6ZV3d8Z,None,None
1134126,#ITintheD384,None,None
1416485,#FelizTarde 👉 https://t.co/i9zJ57T14Y,None,None
...,...,...,...
14939818,#trading #emprende #FelizSabado 👉 https://t....,None,None
15483967,hot bitcoin roomba,None,None
15483968,tremble traffic expressly,None,None
15483969,the buggy brickwork grows this concrete,None,None


In [ ]:
wrong_username_ids = wrong_username_rows.index.tolist()
print("Zeilen mit falschem Typ:", wrong_username_ids)

Zeilen mit falschem Typ: [891075, 891076, 1134125, 1134126, 1416485, 3296300, 3296301, 3296302, 3296303, 3296304, 3296305, 3296306, 3296307, 3296308, 3296309, 3296310, 3296311, 3407877, 3407878, 3407879, 3407880, 3407881, 3407882, 3407883, 3407884, 3407885, 3407886, 3407887, 3407888, 3410683, 3410684, 3410685, 3410686, 3410687, 3410688, 3410689, 3410690, 3410691, 3410692, 3410693, 3410694, 3646111, 3646112, 3884318, 4102517, 4831633, 4831634, 5067033, 5067034, 5067035, 5067036, 5067037, 5181241, 5462725, 5567986, 5800894, 5831097, 6039323, 6039324, 6673666, 6674412, 6676961, 6716830, 6752594, 6755338, 6828536, 6847565, 7057852, 7324540, 7398073, 7424834, 7556694, 7556695, 7685607, 7686349, 7768062, 8111045, 8111046, 8111047, 8126279, 8126280, 8126281, 8612561, 8830967, 8907561, 8925904, 8965799, 9283449, 9283450, 9283451, 9283452, 9506944, 9562014, 9684018, 9726205, 9857644, 9902884, 9916874, 9916875, 9917167, 9917168, 9973498, 9976697, 10132626, 10150111, 10150112, 10199673, 10219080,

In [ ]:
df_raw["username"].apply(type).value_counts()

,count
username,
<class 'str'>,15630308
<class 'NoneType'>,387
<class 'float'>,1


In [ ]:
df_raw["datetime"].apply(type).value_counts()

,count
datetime,
<class 'str'>,15630696


In [ ]:
all_errors = list(set(wrong_text_ids + wrong_username_ids))

In [ ]:
df_raw = df_raw.drop(index=all_errors)

In [ ]:
df_raw.head(100)

,datetime,username,text
0,2021-01-01 23:59:58+00:00,WoodooProd,Crypto adoption in 2021: Top trends and predic...
1,2021-01-01 23:59:53+00:00,YSojourners,Ripple Executive Says US Laws on Crypto Are Va...
2,2021-01-01 23:59:53+00:00,Andama_n,@hextoysofficial | https://t.co/C6LPpuq05n Are...
3,2021-01-01 23:59:51+00:00,YSojourners,FUN and Games: Altcoin Drastically Pumps And D...
4,2021-01-01 23:59:50+00:00,HunnerSa,I bought #Bitcoin at 6k-8k-9k-12k-16k-19k and ...
...,...,...,...
95,2021-01-01 23:54:04+00:00,Cryptic_Oz,@CryptoCobain Use to love this coin when I was...
96,2021-01-01 23:54:02+00:00,FD_SignsOfWar,Checkout the latest #PatriotPowerHour (Episode...
97,2021-01-01 23:53:55+00:00,ConstructnRates,@CryptoTwittter @coin_artist I follow Immutabl...
98,2021-01-01 23:53:50+00:00,camendes,Michael Saylor vs. Keith McCullough | #Bitcoin...


In [ ]:
df_raw= df_raw.sort_values(by="username")

In [ ]:
del all_errors
del wrong_username_ids
del wrong_username_rows
del wrong_text_ids
del wrong_text_rows
gc.collect()

0

In [ ]:
df_raw.to_csv("bitcoin_tweets.csv")

In [ ]:
del df_raw
gc.collect()

0

In [ ]:
chunks = pd.read_csv(
    "bitcoin_tweets.csv",
    chunksize=100_000,
    engine="python",
)

In [ ]:
df_raw = pd.concat(chunks)

In [ ]:
for i, text in enumerate(df_raw["text"]):
  if emoji.emoji_count(text) > 0:
    print(i, df_raw.iloc[i])
    break

4 Unnamed: 0                                             11343039
datetime                              2021-09-21 15:50:13+00:00
username                                           000000000OOK
text          This is not freedom. Bitcoin vision crumbling ...
Name: 4, dtype: object


In [ ]:
cutoff_date = pd.Timestamp('2021-10-19', tz= 'UTC')

In [ ]:
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"])
mask = (df_raw['datetime'] <= cutoff_date)
df_raw = df_raw.loc[mask]

In [ ]:
del chunks
gc.collect()

30

In [ ]:
# Struktur
df_raw["hashtag_count"] = df_raw["text"].str.count(r"#\w+|HASHTAG_")
df_raw["url_count"] = df_raw["text"].str.count(r"URL_")
df_raw["mention_count"] = df_raw["text"].str.count(r"@\w+|MENTION")

# Typen
df_raw["is_retweet"] = df_raw["text"].str.lower().str.startswith("rt ").fillna(False).astype("int8")
df_raw["is_reply"] = df_raw["text"].str.match(r"@\w+").fillna(False).astype("int8")

# Stil
df_raw["tweet_len"] = df_raw["text"].str.len().fillna(False).astype("int16")
df_raw["has_caps"] = df_raw["text"].str.contains(r"\b[A-Z]{3,}\b").fillna(False).astype("int8")
df_raw["has_emoji"] = df_raw["text"].apply(lambda x: int(isinstance(x, str) and emoji.emoji_count(x) > 0)).astype("int8")
df_raw["emoji_count"] = df_raw["text"].apply(lambda x: emoji.emoji_count(x) if isinstance(x, str) else 0).astype("int16")
df_raw["has_exclamation"] = df_raw["text"].str.contains(r"!").fillna(False).astype("int8")

# Verhalten
df_raw["has_link"] = (df_raw["url_count"] > 0).fillna(False).astype("int8")


In [ ]:
user_meta = (
    df_raw
    .groupby("username", observed=True)
    .agg(
        tweet_count=("text", "size"),

        hashtags_per_tweet=("hashtag_count", "mean"),
        urls_per_tweet=("url_count", "mean"),
        mentions_per_tweet=("mention_count", "mean"),

        retweet_ratio=("is_retweet", "mean"),
        reply_ratio=("is_reply", "mean"),

        avg_tweet_len=("tweet_len", "mean"),
        caps_ratio=("has_caps", "mean"),
        emoji_ratio=("has_emoji", "mean"),
        emoji_count_ratio=("emoji_count", "mean"),
        exclamation_ratio=("has_exclamation", "mean"),

        link_tweet_ratio=("has_link", "mean"),
    )
    .reset_index()
)

In [ ]:
user_meta["no_link_tweet_ratio"] = 1.0 - user_meta["link_tweet_ratio"]
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"])
df_raw["date"] = df_raw["datetime"].dt.date

tweets_per_day = (
    df_raw
    .groupby(["username", "date"])
    .size()
    .groupby("username")
    .mean()
)

user_meta["tweets_per_day"] = user_meta["username"].map(tweets_per_day)


In [ ]:
pattern = re.compile(r"@(\w+)")
counts = {}

for text in df_raw["text"]:
    if not isinstance(text, str):
        continue

    for mention in pattern.findall(text):
        counts[mention] = counts.get(mention, 0) + 1

In [ ]:
mentioned_count = (
    pd.DataFrame.from_dict(counts, orient="index", columns=["mentioned_by_others"])
      .rename_axis("username")
      .reset_index()
)

user_meta = user_meta.merge(
    mentioned_count,
    on="username",
    how="left"
)


user_meta["mentioned_by_others"] = user_meta["mentioned_by_others"].fillna(0)

In [ ]:
del pattern
del counts
del mentioned_count
gc.collect()

15

In [ ]:
user_meta.to_csv("meta_data_training.csv")

In [ ]:
user_meta.columns

Index(['username', 'tweet_count', 'hashtags_per_tweet', 'urls_per_tweet',
       'mentions_per_tweet', 'retweet_ratio', 'reply_ratio', 'avg_tweet_len',
       'caps_ratio', 'emoji_ratio', 'emoji_count_ratio', 'exclamation_ratio',
       'link_tweet_ratio', 'no_link_tweet_ratio', 'tweets_per_day',
       'mentioned_by_others'],
      dtype='object')

In [ ]:
user_meta.head(20)

,username,tweet_count,hashtags_per_tweet,urls_per_tweet,mentions_per_tweet,retweet_ratio,reply_ratio,avg_tweet_len,caps_ratio,emoji_ratio,emoji_count_ratio,exclamation_ratio,link_tweet_ratio,no_link_tweet_ratio,tweets_per_day,mentioned_by_others
0,0000000000KAIJU,1,1.000000,0.0,0.000000,0.0,0.000000,95.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.0,1.000000,0.0
1,000000000OOK,6,0.000000,0.0,0.833333,0.0,0.666667,145.500000,0.333333,0.166667,0.166667,0.000000,0.0,1.0,1.200000,3.0
2,00000000FF00,1,0.000000,0.0,1.000000,0.0,1.000000,75.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.0,1.000000,0.0
3,00000000jn,1,0.000000,0.0,1.000000,0.0,1.000000,22.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.0,1.000000,0.0
4,00000001_,1,0.000000,0.0,1.000000,0.0,0.000000,161.000000,1.000000,0.000000,0.000000,0.000000,0.0,1.0,1.000000,0.0
5,0000000dead,2,0.000000,0.0,2.000000,0.0,1.000000,262.500000,0.000000,0.000000,0.000000,0.500000,0.0,1.0,2.000000,0.0
6,00000786L,2,0.000000,0.0,1.500000,0.0,1.000000,58.500000,0.000000,0.500000,0.500000,0.500000,0.0,1.0,2.000000,0.0
7,0000078931463_k,1,0.000000,0.0,0.000000,0.0,0.000000,243.000000,1.000000,0.000000,0.000000,0.000000,0.0,1.0,1.000000,0.0
8,00000ab11,1,8.000000,0.0,1.000000,0.0,0.000000,235.000000,1.000000,0.000000,0.000000,0.000000,0.0,1.0,1.000000,0.0
9,00000nft,9,2.333333,0.0,0.777778,0.0,0.222222,128.111111,0.333333,0.666667,1.444444,0.333333,0.0,1.0,1.285714,0.0


In [ ]:
del user_meta
del df_raw
gc.collect()

28

In [ ]:
def preprocess_bitcoin_tweet(text):
    if pd.isna(text):
        return ""

    # mentions
    text = re.sub(r"@\w+", "@USER", text)

    # urls → nur Domain
    def repl(match):
        url = match.group(0)
        """try:
            domain = urlparse(url).netloc
            if domain:
                return f"HTTPURL_{domain}"
            else:
                return "HTTPURL"
        except:"""
        return "HTTPURL"
    text = re.sub(r"http\S+|www.\S+", repl, text)

    # hashtags
    #text = re.sub(r"#(\w+)", lambda m: f"HASHTAG_{m.group(1).lower()}", text)

    #CASHTAG
    text = re.sub(r"\$[A-Za-z]{1,10}", "CASHTAG", text)

    # multiple spaces → single
    text = " ".join(text.split())

    # lowercase
    #text = text.lower()

    return text

In [ ]:
chunks = []

for chunk in pd.read_csv("bitcoin_tweets.csv", chunksize=100_000):
    chunk["text"] = chunk["text"].map(preprocess_bitcoin_tweet)
    chunks.append(chunk)

df_raw = pd.concat(chunks, ignore_index=True)

In [ ]:
for i, text in enumerate(df_raw["text"]):
  if emoji.emoji_count(text) > 0:
    print(i, df_raw["text"].iloc[i])
    break

4 This is not freedom. Bitcoin vision crumbling by big banks and blockstream. BCH for the win. 💚 HTTPURL


In [ ]:
def preprocess_emojis(text):
    if not isinstance(text, str):
        return ""
    if emoji.emoji_count(text) == 0:
        return text
    # Emojis in Text umwandeln
    text = emoji.demojize(text)#, delimiters=(" ", " "))
    #text = emoji.demojize(text, language='en')
    # Optional: Platzhalter
    #text = re.sub(r":([a-z0-9_]+):", r"EMOJI_\1", text)
    return text

In [ ]:
print(preprocess_emojis("🐋🔥😡📜"))

:whale::fire::pouting_face::scroll:


In [ ]:
df_raw["text"] = df_raw["text"].apply(preprocess_emojis)

In [ ]:
for y, text in enumerate(df_raw["text"]):
  if emoji.emoji_count(text) > 0:
    print(y, df_raw.iloc[y])
    break

In [ ]:
print(df_raw["text"].iloc[i])

This is not freedom. Bitcoin vision crumbling by big banks and blockstream. BCH for the win. :green_heart: HTTPURL


In [ ]:
df_raw["datetime"]= pd.to_datetime(df_raw['datetime'])
df_raw= df_raw.sort_values(by="datetime")

In [ ]:
df_raw.head(100)

,Unnamed: 0,datetime,username,text
808037,6265269,2021-05-19 00:00:00+00:00,BitcoinLifeClub,NoCoiner? A person who doesn't own any #crypto...
9146250,6265270,2021-05-19 00:00:00+00:00,valueofcrypto,#Bitcoin Current Price: $ 42878.74 € 35112.95 ...
760811,6265267,2021-05-19 00:00:01+00:00,BitcoinBellyB,"The price of Spaghetti alla Carbonara is 25,62..."
3683724,6265268,2021-05-19 00:00:01+00:00,NikolaBench,:red_heart_selector:SHAKITA INU !!!! :dog_face...
757865,6265266,2021-05-19 00:00:02+00:00,BitcoinAverage,BitcoinAverage - bitcoin price index - ($ 4294...
...,...,...,...,...
2020616,6265178,2021-05-19 00:01:22+00:00,ECRIndy,Day 1 of #Indy500 practice complete! @USER use...
5067538,6265174,2021-05-19 00:01:25+00:00,WhaleTrades,":chart_decreasing::euro_banknote:$1,982,836 CA..."
3246480,6265173,2021-05-19 00:01:26+00:00,MadnessVs,"@USER &gt;Sell ""Bit coin is good."" &gt;Buy ""Bi..."
6117985,6265171,2021-05-19 00:01:29+00:00,bsv_is_bitcoin,"Mastercard funds BTC ""development"" literally. ..."


In [ ]:
df_raw.to_csv("bitcoin_tweets_clean.csv")

In [ ]:
del df_raw
del chunks

gc.collect()

0